# NumPy — Technical Reference

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Array creation | Build arrays from scratch or data | §1 |
| Array properties & dtypes | Inspect shape, type, memory | §2 |
| Indexing & slicing | Subset arrays in 1D / 2D / nD | §3 |
| Reshaping & stacking | Change dimensions, combine arrays | §4 |
| Math & aggregations | Element-wise ops, reductions | §5 |
| Broadcasting | Operations on arrays of different shapes | §6 |
| Linear algebra | Matrix ops, decompositions, solvers | §7 |
| Random & simulation | Sampling, seeding, distributions | §8 |
| Performance | Views vs copies, vectorization | §9 |

---
## When to Use

| Signal | NumPy method to reach for |
| :--- | :--- |
| Create an array of zeros / ones | `np.zeros()`, `np.ones()`, `np.full()` |
| Evenly spaced values | `np.arange()` (step-based) or `np.linspace()` (count-based) |
| Filter elements by condition | Boolean indexing or `np.where()` |
| Apply condition element-wise | `np.where(cond, x, y)` |
| Aggregate across an axis | `.sum(axis=)`, `.mean(axis=)`, `.max(axis=)` |
| Reshape without copying | `.reshape()` or `.T` |
| Stack arrays side by side | `np.hstack()` or `np.concatenate(axis=1)` |
| Stack arrays top to bottom | `np.vstack()` or `np.concatenate(axis=0)` |
| Matrix multiplication | `A @ B` or `np.dot(A, B)` |
| Solve linear system Ax = b | `np.linalg.solve(A, b)` |
| Generate reproducible random data | `np.random.seed()` then `np.random.*` |
| Check if operation creates a copy | `.base` attribute — `None` means it is a copy |

---
## §1 — Array Creation

Every NumPy workflow starts here. Know which constructor to reach for and what dtype it produces by default.

In [ ]:
import numpy as np

# From Python data
np.array([1, 2, 3])                             # 1D array, dtype inferred
np.array([[1, 2], [3, 4]])                       # 2D array (matrix)
np.array([1, 2, 3], dtype=float)                # force dtype

# Filled arrays
np.zeros((3, 4))                                # 3×4 array of 0.0 (float64 by default)
np.ones((3, 4))                                 # 3×4 array of 1.0
np.full((3, 4), 7)                              # 3×4 array filled with 7
np.eye(4)                                       # 4×4 identity matrix
np.empty((3, 4))                                # uninitialized (fast, values are garbage)

# Range arrays
np.arange(0, 10, 2)                             # [0, 2, 4, 6, 8] — like Python range(), step-based
np.linspace(0, 1, 5)                            # [0, 0.25, 0.5, 0.75, 1.0] — N evenly spaced points

# From existing array
np.zeros_like(arr)                              # same shape and dtype as arr, filled with 0
np.ones_like(arr)
np.copy(arr)                                    # explicit copy

**`np.arange` vs `np.linspace`:**

| | Control | Endpoint included | Use when |
| :--- | :--- | :--- | :--- |
| `np.arange(start, stop, step)` | Step size | ❌ No | You know the step |
| `np.linspace(start, stop, n)` | Count | ✅ Yes | You know how many points |

**Common mistakes:**
- `np.zeros(3, 4)` raises `TypeError` — shape must be a tuple: `np.zeros((3, 4))`
- `np.arange` with float steps accumulates floating point error — use `np.linspace` for floats
- `np.empty` looks initialized but contains arbitrary memory values — never use it when you need zeros

---
## §2 — Array Properties & dtypes

Always check shape and dtype before operating. Silent dtype coercion and shape mismatches are the most common source of bugs.

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])

# Shape and dimensions
a.shape          # (2, 3) — (rows, cols)
a.ndim           # 2 — number of dimensions
a.size           # 6 — total number of elements
len(a)           # 2 — length of first dimension only

# Dtype
a.dtype          # dtype('int64')
a.astype(float)  # cast to float64
a.astype(np.int32)  # cast to 32-bit int (halves memory)

# Memory
a.nbytes         # total bytes: size * itemsize
a.itemsize       # bytes per element

# Common dtypes
# float64  — default for floats, highest precision
# float32  — half the memory, enough for most ML tasks
# int64    — default for integers
# int32    — half the memory
# bool     — True/False, 1 byte per element
# complex128 — complex numbers

**Common mistakes:**
- Mixing int and float arrays — NumPy upcasts silently: `np.array([1, 2]) + np.array([0.5])` returns `float64`
- Using `len(a)` on a 2D array — returns the number of rows, not total elements; use `a.size` for total count
- Forgetting that `astype()` returns a new array — the original is unchanged unless you reassign

---
## §3 — Indexing & Slicing

NumPy slices return **views** (not copies) by default. Modifying a slice modifies the original. Use `.copy()` when you need independence.

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# 1D indexing
a[0]             # first element / first row
a[-1]            # last element / last row
a[1:3]           # rows 1 and 2 (end exclusive)
a[::2]           # every other row

# 2D indexing — [row, col]
a[1, 2]          # row 1, col 2 → 6
a[0, :]          # entire first row → [1, 2, 3]
a[:, 1]          # entire second column → [2, 5, 8]
a[0:2, 1:3]      # submatrix: rows 0–1, cols 1–2

# Boolean indexing — filter elements by condition
a[a > 5]                         # → [6, 7, 8, 9] (1D result, flattened)
a[(a > 2) & (a < 8)]             # AND condition
rows = a[:, 0] > 3               # boolean mask on first column
a[rows]                          # select rows where first col > 3

# Fancy indexing — select with an array of indices
a[[0, 2]]                        # rows 0 and 2 → returns a copy (not a view!)
a[:, [0, 2]]                     # columns 0 and 2
a[[0, 1], [1, 2]]                # elements (0,1) and (1,2) → [2, 6]

# np.where — conditional element selection (returns indices or values)
np.where(a > 5)                  # returns (row_indices, col_indices) tuple
np.where(a > 5, a, 0)            # if > 5 keep value, else 0 (same shape as a)
np.where(a > 5, 'high', 'low')   # string labels

**View vs Copy — critical distinction:**

| Operation | Returns | Modifying changes original? |
| :--- | :--- | :--- |
| Basic slicing `a[0:2]` | View | ✅ Yes |
| Boolean indexing `a[a > 0]` | Copy | ❌ No |
| Fancy indexing `a[[0, 1]]` | Copy | ❌ No |

Check with `arr.base` — if `None`, it's a copy; if it points to another array, it's a view.

**Common mistakes:**
- Expecting a copy from slicing — `b = a[0:2]` then `b[0] = 99` modifies `a` too
- Using `&` / `|` without parentheses in boolean indexing — operator precedence causes wrong results
- `np.where(condition)` with one argument returns indices, not values — use `np.where(cond, x, y)` for value substitution

---
## §4 — Reshaping & Stacking

Reshape is a view when possible (no data copy). Use stacking to combine arrays along existing or new axes.

In [ ]:
a = np.arange(12)                # [0, 1, ..., 11]

# Reshape
a.reshape(3, 4)                  # 3 rows × 4 cols — total elements must match
a.reshape(3, -1)                 # -1 infers the missing dimension (→ 3×4)
a.reshape(-1, 1)                 # column vector: (12, 1)
a.reshape(1, -1)                 # row vector: (1, 12)

# Transpose
m = a.reshape(3, 4)
m.T                              # (4, 3) — swaps all axes
m.transpose(1, 0)                # equivalent for 2D; use for nD with explicit axis order

# Flatten — always returns a copy
m.flatten()                      # (12,) — 1D copy
m.ravel()                        # (12,) — 1D view if possible (prefer over flatten)

# Add / remove dimensions
a[np.newaxis, :]                 # (1, 12) — add axis at position 0
a[:, np.newaxis]                 # (12, 1) — add axis at position 1
np.squeeze(a.reshape(1, 12))     # remove size-1 dimensions → (12,)

# Stacking
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

np.vstack([x, y])                # vertical stack → shape (2, 3)  — stack as rows
np.hstack([x, y])                # horizontal stack → shape (6,)  — concatenate 1D
np.column_stack([x, y])          # stack 1D arrays as columns → shape (3, 2)
np.concatenate([x, y], axis=0)   # general — works on any axis
np.stack([x, y], axis=0)         # new axis stack → shape (2, 3)

# Split
np.split(a, 3)                   # split into 3 equal parts
np.array_split(a, 4)             # allow unequal splits

**`np.stack` vs `np.concatenate` vs `np.vstack`:**

| | Creates new axis? | Use when |
| :--- | :--- | :--- |
| `np.stack` | ✅ Yes | Combining along a *new* axis (e.g. batch dimension) |
| `np.concatenate` | ❌ No | Combining along an *existing* axis |
| `np.vstack` / `np.hstack` | ❌ No | Shorthand for common concatenation patterns |

**Common mistakes:**
- `reshape` fails if element count doesn't match — always verify `a.size` before reshaping
- Confusing `np.stack` (new axis) with `np.concatenate` (existing axis) — stack of two (3,4) arrays gives (2,3,4); concatenate gives (6,4)
- `flatten()` always copies; `ravel()` returns a view when possible — prefer `ravel()` for performance

**`reshape` vs `resize`, and `repeat` vs `tile`:**

| | What it does | Element count | Returns |
| :--- | :--- | :--- | :--- |
| `a.reshape(r, c)` | reinterpret same data in a new shape | must stay the same | view when possible |
| `np.resize(a, shape)` | force a new shape, repeating/truncating data | may change | new array (copy) |
| `np.repeat(a, n)` | repeat **each element** n times | grows | new array |
| `np.tile(a, n)` | repeat the **whole array** n times | grows | new array |


In [3]:
import numpy as np
x = np.array([1, 2, 3])
print("repeat(x, 2):", np.repeat(x, 2))   # each element
print("tile(x, 2)  :", np.tile(x, 2))     # whole array


repeat(x, 2): [1 1 2 2 3 3]
tile(x, 2)  : [1 2 3 1 2 3]


---
## §5 — Math & Aggregations

All operations are element-wise by default. Use the `axis` parameter to reduce along a specific dimension.

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])

# Element-wise arithmetic
a + 10                           # add scalar to every element
a * 2                            # multiply every element
a ** 2                           # square every element
np.sqrt(a)                       # square root element-wise
np.exp(a)                        # e^x element-wise
np.log(a)                        # natural log element-wise
np.abs(a)                        # absolute value

# Aggregations — default: reduce entire array to scalar
a.sum()                          # sum of all elements
a.mean()                         # mean of all elements
a.std()                          # standard deviation
a.min()
a.max()
a.argmin()                       # index of minimum element (flattened)
a.argmax()                       # index of maximum element (flattened)

# Aggregations along an axis
a.sum(axis=0)                    # sum down rows → result shape (3,)  — one value per column
a.sum(axis=1)                    # sum across cols → result shape (2,) — one value per row
a.mean(axis=1, keepdims=True)    # keepdims=True → shape (2,1) instead of (2,) — useful for broadcasting

# Cumulative
a.cumsum(axis=1)                 # running total along each row
a.cumprod(axis=0)                # running product down each column

# Matrix operations
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

A * B                            # element-wise multiplication (NOT matrix multiply)
A @ B                            # matrix multiplication — same as np.dot(A, B)
np.dot(A, B)                     # matrix multiplication (also works for 1D dot product)
np.inner(A, B)                   # inner product
np.outer(A.ravel(), B.ravel())   # outer product

**`axis` mental model:**
- `axis=0` → collapse rows (reduce "down") → result has shape of one row
- `axis=1` → collapse columns (reduce "across") → result has shape of one column
- Think: the axis you name is the axis that **disappears**.

**Common mistakes:**
- `A * B` is element-wise, not matrix multiplication — use `A @ B` for matrix multiply
- Forgetting `keepdims=True` when broadcasting a reduction back against the original — shape mismatch errors
- `np.dot` on 2D arrays is matrix multiply, but on 1D arrays it's dot product — behavior differs by input shape

**`np.dot` vs `@` (matmul) vs `np.inner`:**

| | 1-D inputs | 2-D inputs | ≥3-D (batched) |
| :--- | :--- | :--- | :--- |
| `A @ B` / `np.matmul` | dot product (scalar) | matrix multiply | ✅ broadcasts over batch dims |
| `np.dot(A, B)` | dot product (scalar) | matrix multiply | ❌ NOT batched — sums over a different axis pair |
| `np.inner(A, B)` | dot product (scalar) | contracts **last** axes (= `A @ B.T`) | — |

Prefer `@` — it's the readable, batch-aware default. `np.dot` and `@` agree for 1-D and 2-D but diverge for ≥3-D.

**`==` vs `np.array_equal` vs `np.allclose`:**

| | Returns | Float-safe? | Use when |
| :--- | :--- | :--- | :--- |
| `a == b` | elementwise boolean array | ❌ | You want a per-element mask |
| `np.array_equal(a, b)` | single bool, **exact** | ❌ | Integer / exact equality |
| `np.allclose(a, b)` | single bool, **tolerance** | ✅ | Comparing floats (always use this) |


In [2]:
import numpy as np
# dot vs matmul on batched (3-D) input
P = np.arange(2*2*3).reshape(2,2,3)
Q = np.arange(2*3*2).reshape(2,3,2)
print("(P @ Q).shape   :", (P @ Q).shape)     # batched -> (2,2,2)
print("np.dot(P,Q).shape:", np.dot(P,Q).shape) # NOT batched -> (2,2,2,2)

# float comparison trap
a = np.array([0.1+0.2, 1.0]); b = np.array([0.3, 1.0])
print("a == b        :", a == b)
print("array_equal   :", np.array_equal(a, b))
print("allclose      :", np.allclose(a, b))


(P @ Q).shape   : (2, 2, 2)
np.dot(P,Q).shape: (2, 2, 2, 2)
a == b        : [False  True]
array_equal   : False
allclose      : True


---
## §6 — Broadcasting

Broadcasting lets NumPy operate on arrays of different shapes without copying data. It follows a strict set of rules — misunderstanding them is the most common source of shape errors.

In [ ]:
# Broadcasting rules (applied right-to-left on shapes):
# 1. If arrays have different ndim, prepend 1s to the shorter shape
# 2. Arrays with size 1 along a dimension are stretched to match the other
# 3. If sizes differ and neither is 1, raises ValueError

# Scalar broadcast — always works
a = np.array([[1, 2, 3], [4, 5, 6]])    # shape (2, 3)
a + 10                                   # 10 broadcasts to (2, 3)

# 1D broadcast against 2D
row = np.array([10, 20, 30])             # shape (3,) → treated as (1, 3)
a + row                                  # (2,3) + (1,3) → (2,3) — adds row to each row of a

col = np.array([[10], [20]])             # shape (2, 1)
a + col                                  # (2,3) + (2,1) → (2,3) — adds col to each column of a

# Common pattern: normalize each row (subtract row mean)
a - a.mean(axis=1, keepdims=True)        # (2,3) - (2,1) → broadcasts correctly
# Without keepdims: a.mean(axis=1) has shape (2,) → (2,3) - (2,) raises ValueError

# Outer product via broadcasting
x = np.array([1, 2, 3])                 # shape (3,)
y = np.array([10, 20])                  # shape (2,)
x[np.newaxis, :] * y[:, np.newaxis]     # (1,3) * (2,1) → (2,3)

# Shape compatibility examples
# (2, 3) + (3,)     → OK   — (3,) treated as (1, 3)
# (2, 3) + (2, 1)   → OK   — (2, 1) stretches to (2, 3)
# (2, 3) + (2,)     → ERROR — (2,) treated as (1, 2), incompatible with dim 3
# (2, 3) + (2, 4)   → ERROR — 3 and 4 incompatible, neither is 1

**Broadcasting shape rules — quick reference:**

| Shapes | Compatible? | Result shape |
| :--- | :--- | :--- |
| `(2, 3)` + `(3,)` | ✅ | `(2, 3)` |
| `(2, 3)` + `(2, 1)` | ✅ | `(2, 3)` |
| `(2, 3)` + `(1, 3)` | ✅ | `(2, 3)` |
| `(2, 3)` + `(2,)` | ❌ | — |
| `(2, 3)` + `(2, 4)` | ❌ | — |

**Common mistakes:**
- Subtracting a row mean without `keepdims=True` — result shape is `(n,)` not `(n, 1)`, causing a dimension mismatch
- Assuming broadcasting copies data — it doesn't; it uses strides to simulate the stretched shape
- Shape `(2,)` and shape `(2, 1)` are different — always add `newaxis` or `reshape` to control which axis stretches

---
## §7 — Linear Algebra

All linear algebra lives in `np.linalg`. Critical for quantitative analyst roles and ML. Always check that your matrix is square / full-rank before inverting.

In [ ]:
A = np.array([[2, 1], [5, 3]])   # a GENERAL (non-symmetric) matrix
b = np.array([4, 7])
S = A @ A.T                      # build a SYMMETRIC, positive-definite matrix for eigh/cholesky

# Basic matrix properties
np.linalg.det(A)                 # determinant — 0 means singular (not invertible)
np.linalg.matrix_rank(A)         # rank
np.trace(A)                      # sum of diagonal elements
np.diag(A)                       # extract diagonal as 1D array
np.diag([1, 2, 3])               # create diagonal matrix from 1D array

# Inverse
np.linalg.inv(A)                 # A^{-1} — only for square, non-singular matrices

# Solve linear system Ax = b — preferred over inv(A) @ b (more numerically stable)
x = np.linalg.solve(A, b)        # finds x such that Ax = b

# Norms
np.linalg.norm(b)                # L2 norm (Euclidean) of a vector
np.linalg.norm(A, 'fro')         # Frobenius norm of a matrix
np.linalg.norm(b, ord=1)         # L1 norm

# Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)   # GENERAL matrix — use for any square matrix
eigenvalues, eigenvectors = np.linalg.eigh(S)  # SYMMETRIC matrix only — pass S, not A (see §7 note)

# Singular Value Decomposition — A = U @ diag(S) @ Vt
U, sv, Vt = np.linalg.svd(A)     # sv is 1D array of singular values (descending)

# Least squares — solve over-determined system (more equations than unknowns)
x, residuals, rank, s_ = np.linalg.lstsq(A, b, rcond=None)

# Cholesky decomposition — L @ L.T = S (requires SYMMETRIC positive-definite — pass S, not A)
L = np.linalg.cholesky(S)


**`np.linalg.solve` vs `np.linalg.inv`:**

| | Speed | Numerical stability | Use when |
| :--- | :--- | :--- | :--- |
| `np.linalg.solve(A, b)` | Faster | ✅ Better | Solving Ax = b |
| `np.linalg.inv(A) @ b` | Slower | ❌ Worse | Only when you need A⁻¹ explicitly |

**`np.linalg.eig` vs `np.linalg.eigh`:**

| | Valid input | Output | Use when |
| :--- | :--- | :--- | :--- |
| `np.linalg.eig(A)` | Any square matrix | Possibly complex, unordered | General matrix |
| `np.linalg.eigh(S)` | **Symmetric / Hermitian only** | Real, ascending order | Symmetric matrix — faster & stable |

⚠️ `eigh` reads only one triangle and **assumes** symmetry — it returns a *wrong answer with no error* on a non-symmetric matrix. Only pass it a matrix you know is symmetric (e.g. a covariance matrix, or `A @ A.T`).

**Common mistakes:**
- Calling `np.linalg.inv` on a singular matrix — raises `LinAlgError`; check `np.linalg.det(A) != 0` first
- Passing a **non-symmetric** matrix to `eigh` (or `cholesky`) — `eigh` silently returns wrong eigenvalues; `cholesky` raises `LinAlgError`. Use `eig` for general matrices; reserve `eigh`/`cholesky` for symmetric (positive-definite) input
- Forgetting that `SVD` returns `Vt` (transposed V), not `V` — `A ≈ U @ np.diag(S) @ Vt`, not `@ V`


**Verified — why `eigh` needs a symmetric matrix:**

In [1]:
import numpy as np
A = np.array([[2, 1], [5, 3]])   # non-symmetric
S = A @ A.T                      # symmetric, positive-definite

print("S symmetric? ", np.allclose(S, S.T))
print("eig(S) :", np.sort(np.linalg.eig(S)[0]))   # correct
print("eigh(S):", np.linalg.eigh(S)[0])           # correct (ascending)
print("eig(A) :", np.sort(np.linalg.eig(A)[0]))   # correct for general A
print("eigh(A):", np.linalg.eigh(A)[0], " <- WRONG, A is not symmetric")


S symmetric?  True
eig(S) : [2.56579058e-02 3.89743421e+01]
eigh(S): [2.56579058e-02 3.89743421e+01]
eig(A) : [0.20871215 4.79128785]
eigh(A): [-2.52493781  7.52493781]  <- WRONG, A is not symmetric


---
## §8 — Random & Simulation

Use `np.random.default_rng()` for the modern API (reproducible and thread-safe). The legacy `np.random.seed()` still works but is discouraged in new code.

In [ ]:
# Modern API — recommended
rng = np.random.default_rng(seed=42)     # create a Generator with fixed seed

rng.random((3, 4))                        # uniform [0, 1) — shape (3, 4)
rng.integers(0, 10, size=(3, 4))         # integers in [0, 10)
rng.normal(loc=0, scale=1, size=(100,))  # normal distribution μ=0, σ=1
rng.uniform(low=0, high=1, size=(100,))  # uniform distribution
rng.binomial(n=10, p=0.5, size=(100,))  # binomial distribution
rng.choice([1, 2, 3, 4], size=10)        # sample with replacement
rng.choice([1, 2, 3, 4], size=3, replace=False)  # sample without replacement
rng.shuffle(arr)                          # in-place shuffle
rng.permutation(arr)                      # shuffled copy (original unchanged)

# Legacy API — still common in older code
np.random.seed(42)                        # set global seed
np.random.rand(3, 4)                      # uniform [0, 1)
np.random.randn(3, 4)                     # standard normal
np.random.randint(0, 10, size=(3, 4))    # integers in [0, 10)
np.random.choice([1, 2, 3], size=5)      # sample with replacement

# Monte Carlo example — estimate π
rng = np.random.default_rng(0)
n = 1_000_000
x, y = rng.random(n), rng.random(n)
inside = (x**2 + y**2) < 1
pi_estimate = 4 * inside.mean()          # ≈ 3.1416

**Legacy vs modern API:**

| | `np.random.seed()` (legacy) | `np.random.default_rng()` (modern) |
| :--- | :--- | :--- |
| Thread-safe | ❌ | ✅ |
| Reproducible | ✅ (global state) | ✅ (local state) |
| Recommended | ❌ | ✅ |

**Common mistakes:**
- Using `np.random.seed()` in parallel code — global state causes race conditions; use `default_rng` instead
- Calling `rng.shuffle(arr)` expecting a return value — it modifies `arr` in-place and returns `None`; use `rng.permutation(arr)` for a copy
- `rng.integers(0, 10)` includes 0, excludes 10 — end is exclusive, same as Python `range`

---
## §9 — Performance

NumPy's speed comes from vectorized C operations. The moment you introduce a Python loop, you lose that advantage.

In [ ]:
# Views vs copies — the core memory concept
a = np.arange(12).reshape(3, 4)
b = a[0:2]                       # VIEW — same memory, no copy
c = a[0:2].copy()                # COPY — independent memory
d = a[[0, 1]]                    # COPY — fancy indexing always copies
b.base is a                      # True — b is a view of a
c.base is None                   # True — c is a copy

# Vectorization — always prefer over loops
a = np.arange(1_000_000)

# Slow — Python loop
result = [x**2 for x in a]       # ~200ms

# Fast — vectorized
result = a ** 2                   # ~1ms — 200x faster

# Prefer built-in ufuncs over math module
np.sqrt(a)                        # fast — C implementation
np.log(a)                         # fast
[math.sqrt(x) for x in a]        # slow — Python loop + function call overhead

# Memory layout — C order (row-major) vs Fortran order (col-major)
# NumPy default is C order: iterating over rows is faster than columns
a = np.ones((1000, 1000))
a.sum(axis=1)                     # faster — iterates row by row
a.sum(axis=0)                     # slightly slower — iterates column by column

# Use float32 instead of float64 to halve memory (common in ML)
a = np.ones((1000, 1000), dtype=np.float32)
a.nbytes                          # 4MB vs 8MB for float64

# np.einsum — compact notation for complex contractions
# Equivalent to A @ B:
np.einsum('ij,jk->ik', A, B)     # matrix multiply
# Batch dot product:
np.einsum('bi,bi->b', X, Y)      # dot product per row

**Common mistakes:**
- Calling `.copy()` unnecessarily on every slice — views are free; only copy when you need independence
- Using Python `math` functions inside loops instead of NumPy ufuncs — massive speed penalty
- Storing float64 when float32 suffices — doubles memory usage with no benefit for most ML tasks- `np.vectorize` is a convenience wrapper, **not** a speedup — it loops in Python under the hood; reach for real vectorized ufuncs/ops instead


---
# Decision Guide

```
Creating an array?
├── From existing data                        → np.array()
├── All zeros / ones / constant               → np.zeros(), np.ones(), np.full()
├── Range with fixed step                     → np.arange(start, stop, step)
├── N evenly spaced points                    → np.linspace(start, stop, n)
└── Identity matrix                           → np.eye(n)

Indexing / filtering?
├── Slice by position                         → a[start:stop:step] — returns VIEW
├── Filter by condition                       → a[a > threshold] — returns COPY
├── Select by index list                      → a[[0, 2, 4]] — returns COPY
├── Conditional value substitution            → np.where(cond, x, y)
└── Need to modify without affecting original → a[...].copy()

Changing shape?
├── Reshape to known dimensions               → a.reshape(r, c) or a.reshape(r, -1)
├── Transpose                                 → a.T
├── Flatten to 1D                             → a.ravel() (view) or a.flatten() (copy)
└── Add a dimension                           → a[np.newaxis, :] or a[:, np.newaxis]

Combining arrays?
├── Stack as new rows                         → np.vstack([a, b])
├── Stack as new columns                      → np.column_stack([a, b])
├── Along a new axis                          → np.stack([a, b], axis=n)
└── Along an existing axis                    → np.concatenate([a, b], axis=n)

Aggregating?
├── Entire array                              → a.sum(), a.mean(), etc.
├── Collapse rows (one value per column)      → a.sum(axis=0)
├── Collapse columns (one value per row)      → a.sum(axis=1)
└── Keep original dimensions                  → a.sum(axis=1, keepdims=True)

Linear algebra?
├── Matrix multiplication                     → A @ B
├── Solve Ax = b                              → np.linalg.solve(A, b)
├── Eigenvalues (general)                     → np.linalg.eig(A)
├── Eigenvalues (symmetric)                   → np.linalg.eigh(A)
├── SVD                                       → np.linalg.svd(A)
└── Inverse (use sparingly)                   → np.linalg.inv(A)

Performance?
├── Slow Python loop over array               → replace with vectorized op or ufunc
├── Unexpected data modification via slice    → use .copy() to create independence
├── High memory usage with floats             → try float32 instead of float64
└── Complex multi-axis contraction            → np.einsum()
```